# Data Cleaning — Messy Customer Dataset

This notebook takes a deliberately messy customer/transactions dataset (`messy_customer_data.csv`) and systematically transforms it into a clean, analysis-ready dataset, documenting every decision along the way.

**Workflow:**
1. Load data & produce a data quality report
2. Remove duplicates
3. Standardize inconsistent formatting
4. Remove near-duplicates
5. Handle missing data (column-by-column strategy, justified)
6. Detect and handle outliers
7. Correct data types
8. Before vs. after summary
9. Save the cleaned dataset

In [35]:
import pandas as pd
import numpy as np 
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 130)

## 1. Load and Data Quality Report

In [36]:
# Loading data
data_path = "../data/raw/messy_customer_data.csv"
df_raw = pd.read_csv(data_path)


In [37]:
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(10)

Shape: 3,160 rows x 11 columns


,CustomerID,Name,Gender,Age,SignupDate,City,Country,Email,AnnualIncome,PurchaseAmount,LoyaltyTier
0,CUST-00545,Thomas Smith,MALE,61.0,"October 19, 2024",HOUSTON,Australia,thomas.smith545@example.com,"$92,634.21",123.08,Gold
1,2609,Thomas Martinez,m,NaN,04/08/2024,Philadelphia,USA,thomas.martinez2609@example.com,NaN,161.86,BRONZE
2,1468,Betty Martin,Female,40.0,27/04/2022,San Diego,USA,betty.martin1468@example.com,"$38,843.33",180.80,Gold
3,CUST-01965,James Brown,female,40.0,07/01/2021,Dallas,Canada,james.brown1965@example.com,"$73,680.31",42.60,Gold
4,CUST-01802,Jessica Thomas,female,51.0,05/01/2023,phoenix,United Kingdom,jessica.thomas1802@example.com,"$52,232.12",59.01,gold
5,931,Robert Moore,female,20.0,2022-01-02,Philadelphia,USA,robert.moore931@example.com,"$20,702.83",NaN,silver
6,CUST-01254,Linda Anderson,male,52.0,04/03/2021,Philadelphia,Australia,linda.anderson1254@example.com,"$51,673.93",135.61,bronze
7,CUST-00970,Robert Wilson,Male,67.0,07-Jun-2023,Dallas,USA,robert.wilson970@example.com,"$52,144.07",272.37,GOLD
8,CUST-00073,Linda Smith,Male,59.0,04-Nov-2024,Philadelphia,United Kingdom,linda.smith73@example.com,"$59,243.76",52.48,Gold
9,CUST-02392,Robert Wilson,Female,24.0,NaN,san antonio,Australia,NaN,NaN,187.50,SILVER


In [38]:
df_raw.dtypes


CustomerID            str
Name                  str
Gender                str
Age               float64
SignupDate            str
City                  str
Country               str
Email                 str
AnnualIncome          str
PurchaseAmount    float64
LoyaltyTier           str
dtype: object

In [39]:
def data_quality_report(df, label: str = ""):
    print(f"\n{'='*60}")
    print(f"Data Quality Report: {label}")
    print(f"{'='*60}")
    print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print(f"{'-' * 60}")
    report = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'pct_null' : (df.isnull().mean() * 100).round(2),
        'n_unique': df.nunique()
    })
    return report
    
df_report = data_quality_report(df_raw, 'Raw')
df_report


Data Quality Report: Raw
Rows: 3,160 | Columns: 11
Duplicate Rows: 129
------------------------------------------------------------


,dtype,nulls,pct_null,n_unique
CustomerID,str,0,0.00,3000
Name,str,0,0.00,480
Gender,str,56,1.77,16
Age,float64,102,3.23,72
SignupDate,str,111,3.51,2364
City,str,0,0.00,43
Country,str,0,0.00,4
Email,str,180,5.70,2822
AnnualIncome,str,140,4.43,2812
PurchaseAmount,float64,92,2.91,2780


In [40]:
# Spot data-type issue and value-range anamolies column by column
print(f"{'=' * 60}") 
print("--- 1. Age: Numeric but check range ---")
print(pd.DataFrame(df_raw['Age'].describe()).T)

print(f"{'-' * 60}") 
print(f"--- 2. AnnualIncome: currently stored as TEXT (has $ and commas)---")
print(df_raw['AnnualIncome'].dropna().head(5).tolist())

print(f"{'-' * 60}") 
print("--- 3. PurchaseAmount: numeric but check for negatives ---")
print(f"Min: {df_raw['PurchaseAmount'].min()} | Max: {df_raw['PurchaseAmount'].max()}")
print(f"Negative PurchaseAmount (likely errors or refund) : {(df_raw['PurchaseAmount'] < 0).sum()}")

print(f"{'-' * 60}") 
print("--- 4. SignupDate: mixed formats, currently TEXT ---")
print(df_raw['SignupDate'].dropna().sample(8, random_state=1).tolist())

print(f"{'-' * 60}") 
print("--- Gender: inconsistent categorical labels ---")
print(pd.DataFrame(df_raw['Gender'].value_counts(dropna = False)).T)

print(f"{'-' * 60}") 
print("--- LoyaltyTier: inconsistent casing ---")
print(pd.DataFrame(df_raw['LoyaltyTier'].value_counts(dropna= False)).T)

print(f"{'-' * 60}") 
print("--- City: inconsistent casing / whitespace ---")
print(pd.DataFrame(df_raw['City'].value_counts(dropna= False).head(15)).T)

print(f"{'-' * 60}") 
print("--- CustomerID: inconsistent formatting ---")
print(df_raw['CustomerID'].astype(str).sample(8, random_state= 2).tolist())
print(f"{'=' * 60}")

--- 1. Age: Numeric but check range ---
      count       mean        std  min   25%   50%   75%    max
Age  3058.0  41.734794  19.891083 -5.0  31.0  41.0  50.0  200.0
------------------------------------------------------------
--- 2. AnnualIncome: currently stored as TEXT (has $ and commas)---
['$92,634.21', '$38,843.33', '$73,680.31', '$52,232.12', '$20,702.83']
------------------------------------------------------------
--- 3. PurchaseAmount: numeric but check for negatives ---
Min: -384.78 | Max: 428.2
Negative PurchaseAmount (likely errors or refund) : 39
------------------------------------------------------------
--- 4. SignupDate: mixed formats, currently TEXT ---
['04/03/2021', '2023-12-18', '16/10/2021', '2024-08-07', '29/06/2024', '03/06/2021', '06/01/2021', '28-Nov-2024']
------------------------------------------------------------
--- Gender: inconsistent categorical labels ---
Gender  female    F  male  MALE  Female  Male   Female   m     M   f    Male  FEMALE  other  N

**Data quality findings (raw):**

| Issue type | Column(s) | Detail |
|---|---|---|
| Missing values | `Age`, `SignupDate`, `Email`, `AnnualIncome`, `PurchaseAmount`, `LoyaltyTier` | Ranges from ~3% to ~10% null depending on column |
| Wrong dtype | `AnnualIncome` (text w/ `$`/commas), `SignupDate` (text, 5 different formats), `CustomerID` (mixed string/numeric formatting) | Needs explicit conversion |
| Inconsistent categories | `Gender` (`Male`/`male`/`M`/`MALE`/`m `/etc.), `LoyaltyTier` (`Gold`/`GOLD`/`gold`), `City` (casing + stray whitespace) | Needs standardization to canonical labels |
| Value-range anomalies | `Age` (values like `150`, `-5`, `0`), `AnnualIncome` (a few values like `$5,000,000.00` or negative), `PurchaseAmount` (small number of negative "purchases") | Needs outlier detection & documented handling |
| Duplicates | Whole dataset | Exact duplicate rows plus near-duplicates (same customer, reformatted city) exist and need to be identified |
| ID formatting | `CustomerID` | Some rows carry the `CUST-#####` prefix with/without stray whitespace, others are bare integers |


## 2. Removing Dublicates

In [41]:
df = df_raw.copy()

### Removing exact full row duplicates

In [42]:
# Exact Full Rows duplicates
exact_duplicates = df.duplicated().sum()
print(f"Exact Duplicates row found: {exact_duplicates}")
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop= True)
rows_after = len(df)
print(f"Rows before: {rows_before:,} | Rows After: {rows_after:,} | Rows Removed: {rows_before - rows_after:,}")

Exact Duplicates row found: 129
Rows before: 3,160 | Rows After: 3,031 | Rows Removed: 129


## 3. Standardize inconsistant formating 
### Reformating the Datasets to handle near-duplicates
1. Near-duplicates: same CustomerID + Name + SignupDate but differing only in
2.  formatting (e.g. City casing) are still logically the same record.
3. We standardize key text fields first, then re-check for duplicates on business keys.

**Standardization rules applied below:**
- `Gender` → collapse all casing/whitespace/abbreviation variants into canonical `Male`, `Female`, `Other` labels.
- `LoyaltyTier` → collapse casing variants into canonical `Gold`, `Silver`, `Bronze`.
- `City` → strip stray whitespace, apply title case.
- `CustomerID` → strip whitespace, enforce a single canonical `CUST-#####` format (zero-padded to 5 digits) for every row.
- `SignupDate` → parse the 5 mixed textual date formats into a single proper `datetime64` column.


In [43]:
# Gender re-formating  (Standarization of Gender)
# Gender → collapse all casing/whitespace/abbreviation variants into canonical 'Male', 'Female', 'Other' labels.
def clean_gender(g):
    if pd.isna(g):
        return np.nan
    g = str(g).strip().lower()
    if g in ('m', 'male'):
        return 'Male'
    elif g in ('f', 'female'):
        return 'Female'
    return 'Other'
df['Gender'] = df['Gender'].apply(clean_gender)
print(pd.DataFrame(df['Gender'].value_counts(dropna=False)).T)


Gender  Female  Male  Other  NaN
count     1404  1385    187   55


In [44]:
# Reformating LoyaltyTier (Standarization of LoyaltyTier)
# LoyaltyTier → collapse casing variants into canonical 'Gold', 'Silver', 'Bronze'.
df['LoyaltyTier'] = df['LoyaltyTier'].astype(str).str.strip().str.title()
print(pd.DataFrame(df['LoyaltyTier'].value_counts(dropna=False)).T)

LoyaltyTier  Gold  Bronze  Silver  NaN
count         961     898     879  293


In [45]:
# Reformating City (Standardisation)
# City → strip stray whitespace, apply title case.
df['City'] = df['City'].astype(str).str.strip().str.title()
print(pd.DataFrame(df['City'].value_counts(dropna=False)).T)

City   Chicago  New York  Philadelphia  Austin  Phoenix  San Antonio  Houston  Los Angeles  Dallas  San Diego
count      328       319           313     308      305          301      296          293     285        283


In [46]:
# Reformating CustomerID
# CustomerID → strip whitespace, enforce a single canonical 'CUST-#####' format (zero-padded to 5 digits) for every row.
df['CustomerID'] = df['CustomerID'].astype(str).str.strip().str.upper().str.replace(r"^(\d+)$", lambda m: f"CUST-{int(m.group(1)):05d}", regex=True)
print(df['CustomerID'].astype(str).sample(8, random_state=3).tolist())

['CUST-00840', 'CUST-02303', 'CUST-00820', 'CUST-01455', 'CUST-00960', 'CUST-02876', 'CUST-00574', 'CUST-00294']


In [47]:
# displaying the dateformats to re-formate the date
print(df['SignupDate'].astype(str).sample(40, random_state=4).tolist()) 

['29-Jan-2022', '05/10/2024', '28-Feb-2024', '22-May-2022', 'May 26, 2022', '2023-04-02', '29-Jun-2022', '02-Sep-2023', '09/09/2024', nan, 'August 12, 2024', '2024-04-24', '05/02/2021', '07/24/2023', '20/03/2022', '2023-04-04', 'December 08, 2023', '2021-06-01', '31/01/2024', '2022-05-22', '19/10/2021', '2021-03-12', '2023-09-25', '2021-09-22', '2022-07-01', 'September 30, 2021', '2024-01-19', '23/02/2022', '02/06/2021', '12/19/2023', '06/17/2024', 'September 13, 2022', '04-Sep-2023', '07/22/2023', '03/16/2024', '30-Oct-2021', '2022-01-12', '07/15/2021', '10/25/2023', 'October 28, 2022']


In [48]:
# Reformating SignupDate
# SignupDate → parse the 6 mixed textual date formats into a single proper 'datetime64' column.
def clean_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    for fmt in ('%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%d-%b-%Y', '%b %d, %Y', '%B %d, %Y' ):
        try:
            return pd.to_datetime(val, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(val, errors='coerce') # Last-resort inference
df['SignupDate'] = df['SignupDate'].apply(clean_date)
print(f'Parsed Date: {df['SignupDate'].notna().sum():,} | Still unparsable / Orginally Missing : {df['SignupDate'].isna().sum():,}')
df['SignupDate'].head(10)

Parsed Date: 2,924 | Still unparsable / Orginally Missing : 107


0   2024-10-19
1   2024-04-08
2   2022-04-27
3   2021-07-01
4   2023-05-01
5   2022-01-02
6   2021-04-03
7   2023-06-07
8   2024-11-04
9          NaT
Name: SignupDate, dtype: datetime64[us]

**Note on `%d/%m/%Y` vs `%m/%d/%Y` ambiguity:** Both day-first and month-first slash formats appear in the source data, which is genuinely ambiguous for day ≤ 12 (e.g., `03/04/2022` could be March 4 or April 3). Our parser tries `%m/%d/%Y` before `%d/%m/%Y`, which is correct for the majority U.S.-style records in this dataset; a production cleaning job on a real Kaggle file should confirm the source locale before assuming this ordering, since silently mis-parsing day/month is a common real-world data bug.


## 4. Removing the near-duplicates after reformating data

In [49]:
# Removing dublicates after Reformating 
near_dupe_mask = df.duplicated(subset=['CustomerID', 'Name', 'SignupDate'], keep= 'first')
print(f"Additional near-duplicate rows found (same customer/name/signup date, "
      f"differing only in formatting): {near_dupe_mask.sum()}")
rows_before_near = len(df)
df = df.loc[~near_dupe_mask].reset_index(drop=True)
print(f"Rows Before: {rows_before_near:,} | Rows After: {len(df):,} | Rows Removed: {near_dupe_mask.sum():,}")

Additional near-duplicate rows found (same customer/name/signup date, differing only in formatting): 31
Rows Before: 3,031 | Rows After: 3,000 | Rows Removed: 31


## 5. Missing Data Handling

In [50]:
print("The counting the missing data in each columns: ")
df_missing = df.isnull().sum()
df_missing[df_missing > 0].sort_values(ascending= False)

The counting the missing data in each columns: 


LoyaltyTier       287
Email             174
AnnualIncome      131
SignupDate        103
Age                96
PurchaseAmount     87
Gender             55
dtype: int64

**Column-by-column strategy (and why):**

- **`Age`** → **median imputation**. Age is roughly symmetric with a handful of extreme outliers (handled separately in Section 5), so median is more robust than mean to those outliers. Dropping rows would lose ~5% of the dataset unnecessarily since Age isn't the analysis key.
- **`SignupDate`** → **row deletion for missing values in this field only if also missing elsewhere, otherwise leave as `NaT` after parsing**. A signup date can't be sensibly imputed (no reliable proxy for *when* someone joined), and it's not used as a join key, so leaving it null (properly typed as `NaT`) is more honest than fabricating a date. We do NOT drop these rows since other columns in the same row are still usable.
- **`Email`** → **leave missing (retain as null), do not impute**. There is no valid way to guess an email address; imputing a fake value would actively corrupt the data (e.g. for any future outreach/dedup use). Missingness itself is kept as a signal (e.g., "no email on file").
- **`AnnualIncome`** → **median imputation, after cleaning to numeric**. Income is right-skewed with a few extreme entry-error values (handled in outliers), so median (not mean) is the appropriate central-tendency fill.
- **`PurchaseAmount`** → **median imputation**. This is the core transaction metric; dropping ~3% of rows loses signal unnecessarily, and purchase amounts cluster tightly enough that median imputation won't distort downstream aggregates much. (Rows are relatively low-value here, not identifiers, so imputation is safe.)
- **`LoyaltyTier`** → **mode imputation ("Bronze"), treated as "Unknown" would also be defensible**. We use the most common tier as a neutral fill; since loyalty tier is a categorical convenience field (not core to transaction integrity), mode imputation is reasonable and simple.
- **`CustomerID`, `Name`, `Gender`, `City`, `Country`** → **no missing values found** in this dataset, so no strategy needed — confirmed in the report above.

We deliberately do **not** blanket drop-NA the whole dataframe, since almost every row has *some* missing field and doing so would discard the majority of the data for no good reason.


In [51]:
# Age: Median Imputation
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)
print('Age Median used for imputation is : ', age_median)


Age Median used for imputation is :  40.0


In [52]:
# PurchaseAmount: Median Imputation
purchase_median = df['PurchaseAmount'].median()
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(purchase_median)
print('Purchase Median used for Imputation is: ', purchase_median)

Purchase Median used for Imputation is:  144.19


In [53]:
# LoyaltyTier: Mode Imputation
loyalty_mode = df['LoyaltyTier'].mode()[0]
df['LoyaltyTier'] = df['LoyaltyTier'].fillna(loyalty_mode)
print(f'LoyaltyTier Mode used for imputation is: {loyalty_mode}')

LoyaltyTier Mode used for imputation is: Gold


In [54]:
# AnnualIncome: First clean to numeric (strip $ and ',') then use median Impute
df['AnnualIncome'] = (
    df['AnnualIncome']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .replace('nan', np.nan)
    .astype(float)
)
df['AnnualIncome'].head(5)


0    92634.21
1         NaN
2    38843.33
3    73680.31
4    52232.12
Name: AnnualIncome, dtype: float64

In [55]:
annual_income_median = df['AnnualIncome'].median()
df['AnnualIncome'] = df['AnnualIncome'].fillna(annual_income_median)
print("AnnualIncome Median used for Imputation: ", annual_income_median)


AnnualIncome Median used for Imputation:  61885.19


In [56]:
# SignupDate: parse to datetime (Section 3); missing values intentionally
# left as NaT rather than imputed -- see justification above.
# Email: intentionally left as null -- see justification above.
print("Remaing Nulls after these steps:")
print(df.isnull().sum()[df.isnull().sum() > 0])



Remaing Nulls after these steps:
Gender         55
SignupDate    103
Email         174
dtype: int64


## 6. Outlier Detection (IQR method)

In [57]:
def detect_iqr_outliers(dataframe, columns=None, multiplier=1.5):
    """Return IQR bounds, row masks, and an outlier summary for numeric columns."""
    if columns is None:
        columns = dataframe.select_dtypes(include=np.number).columns.tolist()

    bounds = {} # Store the IQR bounds for each column
    outlier_masks = {}  # Store boolean masks for outliers in each column
    summary = [] # Store summary information for each column

    for column in columns:
        values = dataframe[column].dropna()
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - multiplier * iqr
        upper_bound = q3 + multiplier * iqr
        mask = dataframe[column].lt(lower_bound) | dataframe[column].gt(upper_bound)

        bounds[column] = {
            'Q1': q1,
            'Q3': q3,
            'IQR': iqr,
            'LowerBound': lower_bound.round(2),
            'UpperBound': upper_bound.round(2),
        }
        outlier_masks[column] = mask.fillna(False)
        summary.append({
            'Column': column,
            'LowerBound': lower_bound.round(2),
            'UpperBound': upper_bound.round(2),
            'OutlierCount': int(mask.sum()),
            'OutlierPercentage': round(mask.mean() * 100, 2),
        })

    return bounds, outlier_masks, pd.DataFrame(summary)


numeric_columns = ['Age', 'PurchaseAmount', 'AnnualIncome']
iqr_bounds, iqr_outlier_masks, iqr_summary = detect_iqr_outliers(
    df,
    columns=numeric_columns,
)

print('IQR outlier summary:')
display(iqr_summary)

# Add one combined flag for rows that are outliers in any selected numeric column.
df['HasIQROutlier'] = pd.DataFrame(iqr_outlier_masks, index=df.index).any(axis=1)
print(f"Rows with at least one IQR outlier: {df['HasIQROutlier'].sum()}")

IQR outlier summary:


,Column,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,Age,2.50,78.50,71,2.37
1,PurchaseAmount,-89.10,374.80,45,1.50
2,AnnualIncome,6761.87,117542.08,57,1.90


Rows with at least one IQR outlier: 169


**Outlier decisions (documented):**

- **`Age`** — Values like `150`, `200`, `-5`, `-1`, `0` are **physically impossible**, not just statistically unusual. Decision: **remove rows** where `Age` is outside a sane human bound (`0 < Age <= 100`) rather than cap them, because these look like keystroke/sentinel errors (e.g., `0` as a placeholder), and silently capping them to a boundary value would fabricate plausible-looking but fake ages.
- **`AnnualIncome`** — After conversion to numeric, a very small number of rows sit at extreme values (e.g., ~\$5,000,000 or ~\$999,999,999) that are inconsistent with the rest of the customer-income distribution and look like entry errors (extra digits) rather than genuine ultra-high earners in a general customer file. Decision: **cap at the IQR upper bound** rather than delete, since income is still a directionally usable value for the row (customer/purchase info is otherwise intact) — capping preserves the row while removing the distortion. Negative incomes, if any exist, are removed as impossible.
- **`PurchaseAmount`** — A small number of negative values likely represent **refunds or entry errors**, not legitimate purchases. Decision: **retain but flag** — we add a new boolean `IsNegativePurchase` column rather than deleting or capping, since a negative purchase amount may be a genuine refund record that's meaningful to keep for financial reconciliation; blindly capping to 0 or deleting would destroy that signal. Very large positive values within the IQR outlier range but still plausible (e.g., a big one-time purchase) are **retained as-is**, since large-but-real purchases are valid business data, not errors.


In [58]:
# Age: Remove values below 0 or above 100
age_invalid_mask = (df['Age'] <= 0) | (df['Age'] > 100)
df = df.loc[~age_invalid_mask].copy()

print(f"Removed {age_invalid_mask.sum()} rows with invalid ages")
print(f"Age range after cleaning: {df['Age'].min()} to {df['Age'].max()}")

Removed 61 rows with invalid ages
Age range after cleaning: 16.0 to 82.0


In [59]:
# AnnualIncome: Remove negative values and values above the IQR upper bound
annual_income_upper_bound = iqr_bounds['AnnualIncome']['UpperBound']
negative_income_mask = df['AnnualIncome'] < 0
upper_income_mask = df['AnnualIncome'] > annual_income_upper_bound


negative_income_rows_removed = int(negative_income_mask.sum())
upper_income_rows_removed = int(upper_income_mask.sum())
df = df.loc[~negative_income_mask].copy()
df['AnnualIncome'].clip(upper= annual_income_upper_bound)

print(f"Removed {negative_income_rows_removed} negative AnnualIncome rows")
print(f"Capped {upper_income_rows_removed} AnnualIncome upper outlier rows")
print(f"AnnualIncome range after cleaning: {df['AnnualIncome'].min():,} to {df['AnnualIncome'].max():,}")
print(f"IQR upper bound used: {annual_income_upper_bound:,}")

Removed 13 negative AnnualIncome rows
Capped 42 AnnualIncome upper outlier rows
AnnualIncome range after cleaning: 8,000.0 to 999,999,999.0
IQR upper bound used: 117,542.08


In [60]:
# PurchaseAmount: Flag negative values instead of removing or capping.
purchase_amount_negative_mask = df['PurchaseAmount'] < 0
df['Purchase_Is_Negative'] = purchase_amount_negative_mask
purchase_amount_negative_rows_flagged = int(purchase_amount_negative_mask.sum())

print(f"Flagged {purchase_amount_negative_rows_flagged} negative PurchaseAmount rows")
print(f"PurchaseAmount range: {df['PurchaseAmount'].min():,} to {df['PurchaseAmount'].max():,}")

Flagged 37 negative PurchaseAmount rows
PurchaseAmount range: -384.78 to 428.2


## 7. Datatype Correction

In [61]:
print("--- DataTypes Before Correction: ---")
dtypes_before_correction = df.dtypes
dtypes_before_correction

--- DataTypes Before Correction: ---


CustomerID                         str
Name                               str
Gender                             str
Age                            float64
SignupDate              datetime64[us]
City                               str
Country                            str
Email                              str
AnnualIncome                   float64
PurchaseAmount                 float64
LoyaltyTier                        str
HasIQROutlier                     bool
Purchase_Is_Negative              bool
dtype: object

In [62]:
# Correcting Datatypes
df['Gender'] = df['Gender'].astype('category')
df['Age'] = df['Age'].astype(int)
df['Country'] = df['Country'].astype('category')
df['Email'] = df['Email'].astype(str).replace('nan', np.nan)
df['AnnualIncome'] = df['AnnualIncome'].astype(float).round(2)
df['PurchaseAmount'] = df['PurchaseAmount'].astype(float).round(2)
df['LoyaltyTier'] = df['LoyaltyTier'].astype('category')


In [63]:
print("--- DataTypes After Correction ---")
dtypes_after_correction = df.dtypes

--- DataTypes After Correction ---


**Type corrections applied:** IDs and free-text fields (`CustomerID`, `Name`, `City`, `Email`) are kept as strings (never numeric, to preserve formatting like leading zeros / prefixes and to avoid accidental arithmetic on identifiers). Low-cardinality repeated labels (`Gender`, `Country`, `LoyaltyTier`) are cast to `category` for memory efficiency and to make invalid future values easy to catch. `SignupDate` is a proper `datetime64[ns]`. `Age` is `int` (no fractional people). `AnnualIncome`/`PurchaseAmount` are `float`, rounded to 2 decimals as currency.

## 8. Before vs. After Summary

In [64]:
clean_df = df.copy()

In [65]:
print("IQR outlier summary Before Cleaning: ")
display(iqr_summary)

IQR outlier summary Before Cleaning: 


,Column,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,Age,2.50,78.50,71,2.37
1,PurchaseAmount,-89.10,374.80,45,1.50
2,AnnualIncome,6761.87,117542.08,57,1.90


In [66]:
#numeric_columns = ['Age', 'PurchaseAmount', 'AnnualIncome']
iqr_bounds_cleaned, iqr_outlier_masks_cleaned, iqr_summary_cleaned = detect_iqr_outliers(
    clean_df,
    columns=numeric_columns,
)

print('IQR outlier summary After Cleaning:')
display(iqr_summary_cleaned)
print(f"Rows with at least one IQR outlier after cleaning: {clean_df['HasIQROutlier'].sum()}")

IQR outlier summary After Cleaning:


,Column,LowerBound,UpperBound,OutlierCount,OutlierPercentage
0,Age,2.50,78.50,10,0.34
1,PurchaseAmount,-89.93,375.07,44,1.50
2,AnnualIncome,7330.57,117365.82,42,1.44


Rows with at least one IQR outlier after cleaning: 95


In [67]:
# Before vs. after cleaning summary
source_columns = [
    'CustomerID', 'Name', 'Gender', 'Age', 'City', 'Country',
    'SignupDate', 'Email', 'AnnualIncome', 'PurchaseAmount', 'LoyaltyTier'
]

expected_dtype_checks = {
    'CustomerID': pd.api.types.is_string_dtype,
    'Name': pd.api.types.is_string_dtype,
    'Gender': pd.api.types.is_categorical_dtype,
    'Age': pd.api.types.is_integer_dtype,
    'City': pd.api.types.is_string_dtype,
    'Country': pd.api.types.is_categorical_dtype,
    'SignupDate': pd.api.types.is_datetime64_any_dtype,
    'Email': pd.api.types.is_string_dtype,
    'AnnualIncome': pd.api.types.is_float_dtype,
    'PurchaseAmount': pd.api.types.is_float_dtype,
    'LoyaltyTier': pd.api.types.is_categorical_dtype,
}


def dtype_accuracy(dataframe):
    matched = sum(
        column in dataframe.columns and check(dataframe[column].dtype)
        for column, check in expected_dtype_checks.items()
    )
    total = len(expected_dtype_checks)
    return f'{matched}/{total} ({matched / total:.1%})'


summary_table = pd.DataFrame({
    'Metric': [
        'Row count',
        'Null count',
        'columns with any null',
        'Duplicate count',
        'Dtype accuracy',
    ],
    'Before cleaning': [
        len(df_raw),
        int(df_raw.isna().sum().sum()),
        int((df_raw.isnull().sum() > 0).sum()),
        int(df_raw.duplicated().sum() + near_dupe_mask.sum()),
        dtype_accuracy(df_raw),
    ],
    'After cleaning': [
        len(df),
        int(df.isna().sum().sum()),
        int((df.isnull().sum() > 0).sum()),
        int(df.duplicated().sum()),
        dtype_accuracy(df),
    ],
})

display(summary_table)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6484\3678375976.py:24: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  column in dataframe.columns and check(dataframe[column].dtype)


,Metric,Before cleaning,After cleaning
0,Row count,3160,2926
1,Null count,985,323
2,columns with any null,7,3
3,Duplicate count,160,0
4,Dtype accuracy,5/11 (45.5%),11/11 (100.0%)


**Summary of the cleaning pass:** row count dropped only as much as genuine duplicates/impossible values required, duplicate rows and stray nulls were eliminated from the columns where imputation was appropriate (with `Email` and `SignupDate` nulls intentionally retained per the justification in Section 5), and every column now carries its correct dtype instead of catch-all `str` types. The dataset is now suitable for direct use in aggregation, joins, or modeling without further ad-hoc cleanup.

## 9. Save The Cleaned Dataset

In [68]:
output_path = '../data/cleaned/cleaned_customer_data.csv'
clean_df.to_csv(output_path, index= False)
print(f"Cleaned Dataset saved to '{output_path}'")
print(f"Final Shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns")
clean_df.head(10)

Cleaned Dataset saved to '../data/cleaned/cleaned_customer_data.csv'
Final Shape: 2,926 rows x 13 columns


,CustomerID,Name,Gender,Age,SignupDate,City,Country,Email,AnnualIncome,PurchaseAmount,LoyaltyTier,HasIQROutlier,Purchase_Is_Negative
0,CUST-00545,Thomas Smith,Male,61,2024-10-19,Houston,Australia,thomas.smith545@example.com,92634.21,123.08,Gold,False,False
1,CUST-02609,Thomas Martinez,Male,40,2024-04-08,Philadelphia,USA,thomas.martinez2609@example.com,61885.19,161.86,Bronze,False,False
2,CUST-01468,Betty Martin,Female,40,2022-04-27,San Diego,USA,betty.martin1468@example.com,38843.33,180.80,Gold,False,False
3,CUST-01965,James Brown,Female,40,2021-07-01,Dallas,Canada,james.brown1965@example.com,73680.31,42.60,Gold,False,False
4,CUST-01802,Jessica Thomas,Female,51,2023-05-01,Phoenix,United Kingdom,jessica.thomas1802@example.com,52232.12,59.01,Gold,False,False
5,CUST-00931,Robert Moore,Female,20,2022-01-02,Philadelphia,USA,robert.moore931@example.com,20702.83,144.19,Silver,False,False
6,CUST-01254,Linda Anderson,Male,52,2021-04-03,Philadelphia,Australia,linda.anderson1254@example.com,51673.93,135.61,Bronze,False,False
7,CUST-00970,Robert Wilson,Male,67,2023-06-07,Dallas,USA,robert.wilson970@example.com,52144.07,272.37,Gold,False,False
8,CUST-00073,Linda Smith,Male,59,2024-11-04,Philadelphia,United Kingdom,linda.smith73@example.com,59243.76,52.48,Gold,False,False
9,CUST-02392,Robert Wilson,Female,24,NaT,San Antonio,Australia,NaN,61885.19,187.50,Silver,False,False
